# Eksperimen Tahap 4: Graph Extraction (Two-Tier Chain & Strict Biological Ontology)

Notebook ini berfungsi untuk menarik semua teks **Chunks** yang sudah dibuat di pengujian sebelumnya, lalu mengeksekusi metode ekstraksi *Knowledge Graph* (GraphRAG) menggunakan *Two-Tier Chain* berdasarkan spesifikasi `graphBreakdown.txt`.

In [ ]:
!pip install supabase requests

In [ ]:
import os
import requests
import json
import time
import re
from supabase import create_client, Client

# --- CONFIGURATION ---
SUPABASE_URL = "https://YOUR_PROJECT_REF.supabase.co"
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_ROLE_KEY"
MISTRAL_API_KEY = "YOUR_MISTRAL_API_KEY"
MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"

try:
    supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Supabase terhubung!")
except Exception as e:
    supabase_client = None
    print("Supabase Error:", e)

### Arsitektur Utama (Two-Tier Chain)

In [ ]:
def call_mistral(system_prompt: str, user_prompt: str, temperature: float = 0.1, response_format=None):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {MISTRAL_API_KEY}"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature
    }
    if response_format:
        data["response_format"] = response_format
        
    max_retries = 3
    for i in range(max_retries):
        try:
            resp = requests.post(MISTRAL_URL, headers=headers, json=data)
            resp.raise_for_status()
            return resp.json()['choices'][0]['message']['content'].strip()
        except Exception as e:
            print(f"API Error (percobaan {i+1}): {e}")
            time.sleep(10) # 10 detik delay jika terjadi error limitasi API
    return ""

# --- TIER 1: Peringkasan Khusus Medis ---
def summarize_medical_text(raw_text: str) -> str:
    system_prompt = "You are a strict botanical and pharmacological expert. Summarize the following text by prioritizing and preserving all exact scientific names of plants, chemical compounds, body parts, diseases, and pharmacological verbs. Do not lose any medical context."
    return call_mistral(system_prompt, f"Text to summarize:\n{raw_text}", temperature=0.1)

# --- TIER 2: Ekstraksi Berdasarkan Strict Ontology ---
def extract_biological_graph(summary_text: str) -> list:
    system_prompt = """
You are an expert data extractor for a Knowledge Graph. Extract UNIQUE relationships from the text strictly adhering to the following ontology:

ALLOWED ENTITY TYPES: 'Plant', 'Compound', 'Disease', 'Symptom', 'BodyPart'.

ALLOWED RELATION TYPES: 'contains' (Plant -> Compound), 'cures' (Plant/Compound -> Disease), 'inhibits' (Compound -> Symptom/Disease), 'causes' (Disease -> Symptom).

Return ONLY a valid JSON array of arrays in this exact format:
[["Entity1", "EntityType1", "Relation", "Entity2", "EntityType2"]]

CRITICAL INSTRUCTIONS:
1. Do not invent new relation types.
2. Provide ONLY UNIQUE relationships. NEVER repeat the exact same array element twice.
3. Stop generating once all valid entities are extracted. Limit extraction to maximum 30 unique relations.
"""
    # Kita panggil mode JSON dari Mistral (bisa menggunakan response_format={'type': 'json_object'} tetapi karena kita meminta Array langsung, 
    # seringkali JSON response type memaksa hasil berupa Object dict {}. Oleh sebab itu biarkan default dan kita Parse Regex JSON).
    raw_output = call_mistral(system_prompt, f"Text to extract:\n{summary_text}", temperature=0.2)
    
    try:
        # Membersihkan tanda Markdown backticks (```json ... ```)
        clean_out = re.sub(r"^```json\n?", "", raw_output)
        clean_out = re.sub(r"\n?```$", "", clean_out)
        
        # Mencoba mem Parsing menjadi array Python
        parsed_json = json.loads(clean_out)
        if isinstance(parsed_json, list):
            # Hilangkan Duplikat (Memastikan 100% Unique jika LLM tetap bocor)
            unique_graph = []
            seen = set()
            for item in parsed_json:
                if isinstance(item, list) and len(item) == 5:
                    key = tuple(item)
                    if key not in seen:
                        seen.add(key)
                        unique_graph.append(item)
            return unique_graph
        return []
    except Exception as e:
        print("Gagal parse JSON graph:", e)
        print("Teks Mentah LLM:\n", raw_output)
        return []


### Eksekutor Ekstraksi Graph Batch

In [ ]:
def process_and_save_graphs(chunk_table_name: str, graph_table_name: str, limit_per_journal=3):
    # Mengambil semua jurnal dari chunk_table terkait
    res_journals = supabase_client.table('journals_experiment').select('id, title').execute()
    journals = res_journals.data
    
    if not journals:
        print("Tidak ada jurnal di Database!")
        return
    
    for journal in journals:
        j_id = journal['id']
        j_title = journal['title']
        print(f"\n>>> Memproses Jurnal: {j_title} (Table: {chunk_table_name})")
        
        # Mengambil baris text chunk, kita batasi limit (misal 5) per jurnal agar tidak terlalu lama karena rate limit mistral gratisan
        res_chunks = supabase_client.table(chunk_table_name).select('id, text_content, chunk_index').eq('journal_id', j_id).limit(limit_per_journal).execute()
        chunks = res_chunks.data
        
        total_inserted = 0
        for i, chunk in enumerate(chunks):
            print(f"   - Ekstraksi Chunk {chunk['chunk_index']} / ID: {chunk['id'][:8]}...")
            
            raw_text = chunk['text_content']
            if len(raw_text) < 50:
                print("      (Chunk terlalu pendek, Skip)")
                continue
                
            print("      -> Menjalankan Tier 1 (Medical Summary)...")
            # Tier 1
            summary = summarize_medical_text(raw_text)
            if not summary:
                continue
            
            time.sleep(4) # Menunggu Mistral bernafas 4 detik sebelum lanjut ke Tier 2
            
            print("      -> Menjalankan Tier 2 (Strict Ontology JSON)...")
            # Tier 2
            graph_data = extract_biological_graph(summary)
            
            # Insert Graph array ke tabel target
            inserted = 0
            for rel in graph_data:
                # Validasi struktur Array ["E1", "T1", "R", "E2", "T2"]
                if len(rel) == 5:
                    try:
                        supabase_client.table(graph_table_name).insert({
                            "journal_id": j_id,
                            "chunk_id": chunk['id'],
                            "entity_1": rel[0],
                            "type_1": rel[1],
                            "relation": rel[2],
                            "entity_2": rel[3],
                            "type_2": rel[4]
                        }).execute()
                        inserted += 1
                    except Exception as e:
                        print(f"      Insert error: {e}")
            
            print(f"      > Berhasil mengekstrak dan menyimpan: {inserted} Hubungan/Relasi Graph.\n")
            total_inserted += inserted
            
            time.sleep(5) # Delay 5 detik antar CHUNK agar limit API Free-Tier tenang
            
        print(f"[!] {chunk_table_name} - Selesai jurnal '{j_title}'. Total Hubungan Ditemukan: {total_inserted}")
        
        time.sleep(10) # Istirahat panjang selama 10 detik setiap 1 jurnal selesai


---

### 1. Proses Semua Chunk berjenis Character (`chunk_character`)

In [ ]:
# Eksekusi Karakter
# Set limit 3 array chunk pertama per jurnal sebagai basis riset komparasi.

process_and_save_graphs("chunk_character", "graph_character", limit_per_journal=3)

### 2. Proses Semua Chunk berjenis Word (`chunk_word`)

In [ ]:
# Eksekusi Kata/Token
process_and_save_graphs("chunk_word", "graph_word", limit_per_journal=3)

### 3. Proses Semua Chunk berjenis Sentence (`chunk_sentence`)

In [ ]:
# Eksekusi Kalimat
process_and_save_graphs("chunk_sentence", "graph_sentence", limit_per_journal=3)